# 微调 Open AI 模型

本笔记本基于 Open AI 提供的当前[微调](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)文档中的指导。

> **注意：** 本笔记本的示例输出是基于 `gpt-3.5-turbo` 生成的，该模型现在在 Microsoft Foundry 的 Azure OpenAI 中已同时停止推理和微调使用（并且 OpenAI 已直接弃用）。下面的演练和概念依然准确，但如果你今天开始新的微调任务，请针对当前受支持的模型，例如 `gpt-4o-mini` 或 `gpt-4.1-mini`。有关当前可微调模型的完整列表，请参见[微调模型列表](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)。

微调通过使用针对特定用例或场景的额外数据和上下文重新训练基础模型，提升其在你的应用中的性能。请注意，提示工程技术如 _few shot learning_（少量实例学习）和 _retrieval augmented generation_（检索增强生成）允许你用相关数据增强默认提示以提高质量，但这些方法受限于目标基础模型的最大令牌窗口大小。

通过微调，我们实际上是用所需数据重新训练模型本身（允许我们使用远超过最大令牌窗口容量的示例）——并部署一个 _定制_ 版本的模型，在推理时无需再提供实例。这不仅提升了我们设计提示的效果（我们有更多灵活性将令牌窗口用于其他用途），也可能降低我们的成本（减少推理时需发送给模型的令牌数）。

微调有四个步骤：
1. 准备训练数据并上传。
1. 运行训练作业获得微调模型。
1. 评估微调模型并迭代提升质量。
1. 满意后部署微调模型进行推理。

请注意，并非所有基础模型都支持微调——请查看[OpenAI文档](https://platform.openai.com/docs/guides/fine-tuning/what-models-can-be-fine-tuned?WT.mc_id=academic-105485-koreyst)获取最新信息。你也可以对之前微调过的模型进行再次微调。在本教程中，我们将使用 `gpt-35-turbo` 作为微调的目标基础模型（参见上文备注了解当前受支持的替代模型）。

---


### 第 1.1 步：准备您的数据集

让我们构建一个聊天机器人，通过用五行诗回答关于元素的问题，帮助您理解元素周期表。在_本_简单教程中，我们只会创建一个数据集，用几个示例响应来训练模型，展示数据的预期格式。在实际应用中，您需要创建一个包含更多示例的数据集。如果存在适合您应用领域的开放数据集，您也可以使用它并重新格式化以用于微调。

由于我们专注于 `gpt-35-turbo` 并且寻求单轮响应（聊天补全），我们可以使用[这个建议的格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)创建示例，符合 OpenAI 聊天补全的要求。如果您期望多轮对话内容，则应使用包含 `weight` 参数的[多轮示例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，该参数用于标示哪些消息应被用于微调过程（或不使用）。

本教程这里我们将使用较简单的单轮格式。数据采用[jsonl 格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一个记录，每条记录是一个 JSON 格式的对象。下面的片段展示了 2 条记录作为示例——完整示例集（10 个示例）请参见[training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)，这是我们用于微调教程的样例。**注意：** 每条记录_必须_定义在单行内（不能分跨多行，如格式化 JSON 文件中常见的那样）

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在实际应用中，您将需要多得多的示例集才能获得良好效果——权衡点在于响应质量与微调所需时间／费用。我们使用小样本集以便快速完成微调，演示流程。更复杂的微调教程可见[OpenAI Cookbook 示例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)。


---

### 步骤 1.2 上传您的数据集

使用文件 API 上传数据，[详见此处](https://platform.openai.com/docs/guides/fine-tuning/upload-a-training-file)。请注意，要运行此代码，您必须先完成以下步骤：
 - 安装 `openai` Python 包（确保使用版本 >=0.28.0 以获得最新功能）
 - 设置环境变量 `OPENAI_API_KEY` 为您的 OpenAI API 密钥
欲了解更多信息，请参阅本课程提供的[安装指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)。

现在，运行代码以从本地 JSONL 文件创建一个用于上传的文件。


In [24]:
from openai import OpenAI
client = OpenAI()

ft_file = client.files.create(
  file=open("./training-data.jsonl", "rb"),
  purpose="fine-tune"
)

print(ft_file)
print("Training File ID: " + ft_file.id)

FileObject(id='file-JdAJcagdOTG6ACNlFWzuzmyV', bytes=4021, created_at=1715566183, filename='training-data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Training File ID: file-JdAJcagdOTG6ACNlFWzuzmyV


---

### 第2.1步：使用SDK创建微调作业


In [25]:
from openai import OpenAI
client = OpenAI()

ft_filejob = client.fine_tuning.jobs.create(
  training_file=ft_file.id, 
  model="gpt-3.5-turbo"
)

print(ft_filejob)
print("Fine-tuning Job ID: " + ft_filejob.id)

FineTuningJob(id='ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', created_at=1715566184, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-EZ6ag0n0S6Zm8eV9BSWKmE6l', result_files=[], seed=830529052, status='validating_files', trained_tokens=None, training_file='file-JdAJcagdOTG6ACNlFWzuzmyV', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)
Fine-tuning Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh


---

### 步骤 2.2：检查作业状态

这里有一些你可以使用 `client.fine_tuning.jobs` API 做的事情：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最近的 n 个微调作业
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 获取特定微调作业的详细信息
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消一个微调作业
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<b>)` - 列出作业的最多 n 条事件
- `client.fine_tuning.jobs.create(model="gpt-35-turbo", training_file="your-training-file.jsonl", ...)`

过程的第一步是 _验证训练文件_ 以确保数据格式正确。


In [26]:
from openai import OpenAI
client = OpenAI()

# List 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(ft_filejob.id)

# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_filejob.id, limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-GkWiDgZmOsuv4q5cSTEGscY6', created_at=1715566184, level='info', message='Validating training file: file-JdAJcagdOTG6ACNlFWzuzmyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-3899xdVTO3LN7Q7LkKLMJUnb', created_at=1715566184, level='info', message='Created fine-tuning job: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [30]:
# Once the training data is validated
# Track the job status to see if it is running and when it is complete
from openai import OpenAI
client = OpenAI()

response = client.fine_tuning.jobs.retrieve(ft_filejob.id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh
Status: running
Trained Tokens: None


---

### 步骤 2.3：跟踪事件以监控进度


In [44]:
# You can also track progress in a more granular way by checking for events
# Refresh this code till you get the `The job has successfully completed` message
response = client.fine_tuning.jobs.list_events(ft_filejob.id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 85/100: training loss=0.14
Step 86/100: training loss=0.00
Step 87/100: training loss=0.00
Step 88/100: training loss=0.07
Step 89/100: training loss=0.00
Step 90/100: training loss=0.00
Step 91/100: training loss=0.00
Step 92/100: training loss=0.00
Step 93/100: training loss=0.00
Step 94/100: training loss=0.00
Step 95/100: training loss=0.08
Step 96/100: training loss=0.05
Step 97/100: training loss=0.00
Step 98/100: training loss=0.00
Step 99/100: training loss=0.00
Step 100/100: training loss=0.00
Checkpoint created at step 80 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyyF2:ckpt-step-80
Checkpoint created at step 90 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyzhK:ckpt-step-90
New fine-tuned model created: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz
The job has successfully completed


### 第2.4步：在OpenAI控制台查看状态


您还可以通过访问 OpenAI 网站并浏览平台的 _Fine-tuning_ 部分来查看状态。这将显示当前任务的状态，并让您跟踪之前任务执行的历史记录。在这张截图中，您可以看到之前的执行失败了，第二次运行成功了。作为背景说明，第一次运行时使用的 JSON 文件中记录格式错误 — 修正后，第二次运行成功完成，并使模型可供使用。

![Fine-tuning job status](../../../../../translated_images/zh-CN/fine-tuned-model-status.563271727bf7bfba.webp)


您还可以通过在可视化仪表板中向下滚动查看状态消息和指标，如下所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-CN/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-CN/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 步骤 3.1：在代码中检索 ID 并测试微调模型


In [46]:
# Retrieve the identity of the fine-tuned model once ready
response = client.fine_tuning.jobs.retrieve(ft_filejob.id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)

Fine-tuned Model ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz


In [ ]:
# You can then use that model to generate completions from the SDK as shown
# Or you can load that model into the OpenAI Playground (in the UI) to validate it from there.
from openai import OpenAI
client = OpenAI()

completion = client.responses.create(
  model=fine_tuned_model_id,
  input=[
    {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Strontium"},
  ],
  store=False,
)
print(completion.output_text)


ChatCompletionMessage(content="Strontium, a metal so bright - It's in fireworks, a dazzling sight - It's in bones, you see - And in tea, it's the key - It's the fortieth, so pure, that's the right", role='assistant', function_call=None, tool_calls=None)


---

### 第3.2步：在Playground中加载并测试微调模型

现在您可以通过两种方式测试微调模型。首先，您可以访问Playground，并使用模型下拉菜单从列出的选项中选择您新微调的模型。另一种选择是在微调面板中使用“Playground”选项（见上方截图），它会启动以下_对比_视图，显示基础模型和微调模型版本并排，以便快速评估。

![微调作业状态](../../../../../translated_images/zh-CN/fine-tuned-playground-compare.56e06f0ad8922016.webp)

只需填写训练数据中使用的系统上下文并提供测试问题。您会注意到两边都会更新相同的上下文和问题。运行对比后，您将看到它们输出之间的差异。_注意微调模型如何按照您示例中提供的格式呈现响应，而基础模型则仅遵循系统提示_。

![微调作业状态](../../../../../translated_images/zh-CN/fine-tuned-playground-launch.5a26495c983c6350.webp)

您会注意到对比还提供了每个模型的令牌数和推理所用时间。**这个具体示例比较简单，主要展示流程，而非真正反映实际数据集或场景**。您可能会注意到两个样例显示的令牌数相同（系统上下文和用户提示相同），但微调模型推理时间更长（自定义模型）。

在真实场景中，您不会使用像这样的玩具示例，而是对真实数据（例如客户服务的产品目录）进行微调，在那种情况下响应质量会更加明显。在_那个_场景下，使用基础模型获得等效响应质量将需要更多自定义提示工程，这会增加令牌使用量并可能增加推理相关的处理时间。_要尝试这一点，请查看OpenAI Cookbook中的微调示例开始操作。_

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
